# EDA: สำรวจ UNSW-NB15 ก่อนแปลงข้อมูล

โค้ดของขั้น EDA อยู่ใน notebook นี้โดยตรง ไม่เรียกฟังก์ชันสำรวจจากไฟล์ Python ภายนอก

ลำดับงาน: ระบุตำแหน่งไฟล์ → อ่านตัวอย่างไม่เกิน 20,000 แถวต่อไฟล์ → ตรวจ schema/ชนิดข้อมูล/missing/ค่าที่ไม่ซ้ำ → เปรียบเทียบ Train/Test → บันทึกรายงาน

> ต้องวาง official CSV ใน `raw/` ก่อน ตัวเลข missing และค่าที่ไม่ซ้ำเป็นผลจาก sample ไม่ใช่สถิติเต็มชุดข้อมูล

## 1. หาไฟล์ต้นทางและตั้งค่าการสำรวจ

ตรวจว่ามีไฟล์ official Train/Test ก่อนอ่าน หากขาดไฟล์จะหยุดพร้อมบอกตำแหน่งที่ต้องวาง

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

current = Path.cwd().resolve()
project_root = next((path for path in (current, *current.parents)
                     if (path / 'data-pipeline' / 'UNSW-NB15' / 'README.md').is_file()), None)
if project_root is None:
    raise FileNotFoundError('เปิด notebook จากภายในโฟลเดอร์โปรเจกต์ CSI_ML_Final')

dataset_dir = project_root / 'data-pipeline' / 'UNSW-NB15'
raw_dir = dataset_dir / 'raw'
output_dir = dataset_dir / 'output' / 'reports'
sample_rows = 20_000
source_files = {
    'train': raw_dir / 'UNSW_NB15_training-set.csv',
    'test': raw_dir / 'UNSW_NB15_testing-set.csv',
}
missing_files = [path.name for path in source_files.values() if not path.is_file()]
if missing_files:
    raise FileNotFoundError(
        f'ไม่พบไฟล์ {missing_files} ใน {raw_dir}; ดาวน์โหลดจากแหล่งที่ได้รับอนุญาตแล้ววางไฟล์ก่อน'
    )
print('โฟลเดอร์ raw:', raw_dir)
for split_name, path in source_files.items():
    print(f'{split_name}: {path.name} ({path.stat().st_size:,} bytes)')

โฟลเดอร์ raw: C:\Users\Windows 10\Desktop\CSI_ML_Final\data-pipeline\UNSW-NB15\raw
train: UNSW_NB15_training-set.csv (32,293,015 bytes)
test: UNSW_NB15_testing-set.csv (15,380,797 bytes)


## 2. สำรวจ schema และคุณภาพเบื้องต้น

อ่าน sample แล้วนับ missing และ unique ต่อคอลัมน์ การดู sample ช่วยตรวจรูปแบบได้เร็ว แต่ผลไม่แทนสถิติทุกแถว และ EDA ไม่แก้ไฟล์ raw

In [2]:
sample_frames = {}
schema_rows = []

for split_name, path in source_files.items():
    sample = pd.read_csv(path, nrows=sample_rows, low_memory=False)
    sample.columns = [str(name).replace('\ufeff', '').strip() for name in sample.columns]
    if len(set(sample.columns)) != len(sample.columns):
        raise ValueError(f'{path.name}: ชื่อคอลัมน์ซ้ำหลังตัด BOM/ช่องว่าง')
    sample_frames[split_name] = sample

    for column in sample.columns:
        schema_rows.append({
            'split': split_name, 'file': path.name, 'column': column,
            'dtype_in_sample': str(sample[column].dtype),
            'sample_rows': len(sample),
            'missing_in_sample': int(sample[column].isna().sum()),
            'unique_in_sample': int(sample[column].nunique(dropna=True)),
        })
schema_report = pd.DataFrame(schema_rows)
display(schema_report.head(100))
print(f'ตรวจ {len(sample_frames)} split, {len(schema_report)} รายการคอลัมน์')

,split,file,column,dtype_in_sample,sample_rows,missing_in_sample,unique_in_sample
0,train,UNSW_NB15_training-set.csv,id,int64,20000,0,20000
1,train,UNSW_NB15_training-set.csv,dur,float64,20000,0,14566
2,train,UNSW_NB15_training-set.csv,proto,str,20000,0,5
3,train,UNSW_NB15_training-set.csv,service,str,20000,0,8
4,train,UNSW_NB15_training-set.csv,state,str,20000,0,7
...,...,...,...,...,...,...,...
85,test,UNSW_NB15_testing-set.csv,ct_src_ltm,int64,20000,0,50
86,test,UNSW_NB15_testing-set.csv,ct_srv_dst,int64,20000,0,57
87,test,UNSW_NB15_testing-set.csv,is_sm_ips_ports,int64,20000,0,2
88,test,UNSW_NB15_testing-set.csv,attack_cat,str,20000,0,10


ตรวจ 2 split, 90 รายการคอลัมน์


## 3. เปรียบเทียบ Train/Test และตรวจฟิลด์ที่ต้องใช้

pipeline จะค้นฟิลด์จากชื่อใน header ไม่ใช้ตำแหน่ง ถ้าฟิลด์ที่ต้องใช้หายไป ให้ตรวจรุ่น/ชื่อคอลัมน์ก่อน harmonization ส่วน label summary เป็นเพียงผลนับจาก sample เพื่อสำรวจเท่านั้น

In [3]:
train_columns = set(sample_frames['train'].columns)
test_columns = set(sample_frames['test'].columns)
required_columns = {'dur', 'proto', 'sbytes', 'dbytes', 'label', 'attack_cat'}
comparison = pd.DataFrame([
    {'check': 'จำนวนคอลัมน์เท่ากัน', 'train': len(train_columns), 'test': len(test_columns),
     'passed': len(train_columns) == len(test_columns)},
    {'check': 'ชื่อคอลัมน์ตรงกัน', 'train': len(train_columns), 'test': len(test_columns),
     'passed': train_columns == test_columns},
])
display(comparison)
print('ฟิลด์จำเป็นที่ไม่พบใน Train:', sorted(required_columns - train_columns))
print('ฟิลด์จำเป็นที่ไม่พบใน Test:', sorted(required_columns - test_columns))

label_rows = []
for split_name, sample in sample_frames.items():
    if {'label', 'attack_cat'}.issubset(sample.columns):
        summary = (sample.groupby(['label', 'attack_cat'], dropna=False).size()
                  .rename('sample_rows').reset_index())
        summary.insert(0, 'split', split_name)
        label_rows.append(summary)
label_sample_summary = (pd.concat(label_rows, ignore_index=True)
                        if label_rows else pd.DataFrame())
display(label_sample_summary.head(30))

,check,train,test,passed
0,จำนวนคอลัมน์เท่ากัน,45,45,True
1,ชื่อคอลัมน์ตรงกัน,45,45,True


ฟิลด์จำเป็นที่ไม่พบใน Train: []
ฟิลด์จำเป็นที่ไม่พบใน Test: []


,split,label,attack_cat,sample_rows
0,train,0,Normal,20000
1,test,0,Normal,243
2,test,1,Analysis,151
3,test,1,Backdoor,49
4,test,1,DoS,2922
5,test,1,Exploits,5723
6,test,1,Fuzzers,1011
7,test,1,Generic,7989
8,test,1,Reconnaissance,1737
9,test,1,Shellcode,155


## 4. บันทึกรายงาน EDA

รายงานช่วยย้อนดูว่าไฟล์ Train/Test มี schema ที่สอดคล้องและฟิลด์จำเป็นอยู่ครบหรือไม่ หากพบ missing หรือชื่อคอลัมน์ไม่ตรง ให้ตรวจต้นทางก่อนกำหนดกฎ cleansing

In [4]:
output_dir.mkdir(parents=True, exist_ok=True)
schema_path = output_dir / 'eda_schema_report.csv'
comparison_path = output_dir / 'eda_split_comparison.csv'
labels_path = output_dir / 'eda_label_sample_summary.csv'
schema_report.to_csv(schema_path, index=False)
comparison.to_csv(comparison_path, index=False)
label_sample_summary.to_csv(labels_path, index=False)
print('บันทึกรายงานแล้ว:')
for report_path in (schema_path, comparison_path, labels_path):
    print('-', report_path.relative_to(project_root))

บันทึกรายงานแล้ว:
- data-pipeline\UNSW-NB15\output\reports\eda_schema_report.csv
- data-pipeline\UNSW-NB15\output\reports\eda_split_comparison.csv
- data-pipeline\UNSW-NB15\output\reports\eda_label_sample_summary.csv
